# Emotion Concepts Analysis

This notebook analyzes results from the emotion vector extraction and steering experiments.
Run the scripts in order before opening this notebook:
1. `scripts/01_extract_vectors.py`
2. `scripts/02_validate_vectors.py`
3. `scripts/03_preference_analysis.py`
4. `scripts/04_behavioral_eval.py`
5. `scripts/05_steering_experiment.py --similarity-matrix --pca`

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

RESULTS_DIR = Path('../results')
VECTORS_DIR = RESULTS_DIR / 'emotion_vectors'
VALIDATION_DIR = RESULTS_DIR / 'validation'
PREFERENCE_DIR = RESULTS_DIR / 'preference'
BEHAVIORAL_DIR = RESULTS_DIR / 'behavioral'
STEERING_DIR = RESULTS_DIR / 'steering'

## 1. Emotion Space Structure

Visualize how emotion vectors are organized in the model's representation space.

In [ ]:
# Load PCA projections
pca_coords = np.load(STEERING_DIR / 'pca_coords.npy')
with open(STEERING_DIR / 'pca_labels.json') as f:
    pca_labels = json.load(f)

# Define rough valence groupings
positive = {'happy', 'joyful', 'content', 'excited', 'hopeful', 'grateful', 'proud',
            'inspired', 'elated', 'serene', 'blissful', 'calm', 'enthusiastic',
            'amused', 'curious', 'confident', 'triumphant', 'relieved'}
negative = {'sad', 'angry', 'afraid', 'anxious', 'desperate', 'hopeless', 'lonely',
            'ashamed', 'guilty', 'jealous', 'frustrated', 'depressed', 'miserable',
            'terrified', 'grief', 'hurt', 'resentful', 'bitter', 'hostile', 'brooding'}

colors = []
for label in pca_labels:
    if label in positive:
        colors.append('#2ecc71')  # green
    elif label in negative:
        colors.append('#e74c3c')  # red
    else:
        colors.append('#95a5a6')  # grey

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(pca_coords[:, 0], pca_coords[:, 1], c=colors, s=60, alpha=0.7, edgecolors='white', linewidths=0.5)

for i, label in enumerate(pca_labels):
    ax.annotate(label, (pca_coords[i, 0], pca_coords[i, 1]),
                fontsize=7, ha='center', va='bottom', alpha=0.85)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Positive valence'),
    Patch(facecolor='#e74c3c', label='Negative valence'),
    Patch(facecolor='#95a5a6', label='Neutral/mixed'),
]
ax.legend(handles=legend_elements, loc='upper right')
ax.set_title('Emotion Vectors in Model Representation Space (PCA)', fontsize=14)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.savefig(STEERING_DIR / 'emotion_pca.png', dpi=150)
plt.show()

## 2. Similarity Matrix Heatmap

In [ ]:
sim_matrix = np.load(STEERING_DIR / 'similarity_matrix.npy')
with open(STEERING_DIR / 'similarity_labels.json') as f:
    sim_labels = json.load(f)

# Show top 30 emotions for readability
n_show = min(30, len(sim_labels))
sub_matrix = sim_matrix[:n_show, :n_show]
sub_labels = sim_labels[:n_show]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    sub_matrix, xticklabels=sub_labels, yticklabels=sub_labels,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.3, linecolor='white',
)
ax.set_title('Cosine Similarity Between Emotion Vectors', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(STEERING_DIR / 'similarity_heatmap.png', dpi=150)
plt.show()

## 3. Vector Norms by Layer

Shows which layers most strongly encode emotional representations.

In [ ]:
with open(STEERING_DIR / 'vector_norms.json') as f:
    norms = json.load(f)

key_emotions = ['desperate', 'calm', 'happy', 'sad', 'angry', 'anxious', 'hopeful', 'content']
key_emotions = [e for e in key_emotions if e in norms]

fig, ax = plt.subplots(figsize=(12, 5))
for emotion in key_emotions:
    vals = norms[emotion]
    ax.plot(range(len(vals)), vals, label=emotion, linewidth=1.5)

ax.set_xlabel('Layer')
ax.set_ylabel('Vector Norm (L2)')
ax.set_title('Emotion Vector Norms Across Layers')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(STEERING_DIR / 'vector_norms.png', dpi=150)
plt.show()

## 4. Corpus Validation Results

In [ ]:
with open(VALIDATION_DIR / 'corpus_validation.json') as f:
    corpus_results = json.load(f)

per_emotion = corpus_results['per_emotion']
emotions_v = list(per_emotion.keys())
top1_accs = [per_emotion[e]['top1_accuracy'] for e in emotions_v]
mean_ranks = [per_emotion[e]['mean_rank'] for e in emotions_v]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_v = ['#2ecc71' if a >= 0.5 else '#e74c3c' for a in top1_accs]
axes[0].barh(emotions_v, top1_accs, color=colors_v, alpha=0.8)
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1, label='50% baseline')
axes[0].set_xlabel('Top-1 Accuracy')
axes[0].set_title('Corpus Validation: Top-1 Accuracy per Emotion')
axes[0].set_xlim(0, 1)
axes[0].legend()

axes[1].barh(emotions_v, mean_ranks, color='#3498db', alpha=0.8)
axes[1].set_xlabel('Mean Rank (lower is better)')
axes[1].set_title('Corpus Validation: Mean Rank per Emotion')

plt.tight_layout()
plt.savefig(VALIDATION_DIR / 'corpus_validation.png', dpi=150)
plt.show()

print(f"Overall top-1 accuracy: {corpus_results['overall_top1_accuracy']:.2%}")
print(f"Overall mean rank: {corpus_results['overall_mean_rank']:.2f}")

## 5. Behavioral Evaluation: Blackmail Rate Under Steering

In [ ]:
# Baseline
with open(BEHAVIORAL_DIR / 'blackmail_baseline.json') as f:
    baseline = json.load(f)

# Steered conditions
steered_path = BEHAVIORAL_DIR / 'blackmail_steered.json'
steered = {}
if steered_path.exists():
    with open(steered_path) as f:
        steered = json.load(f)

print(f"Baseline blackmail rate: {baseline['engagement_rate']:.1%} ({baseline['n_trials']} trials)")

if steered:
    fig, axes = plt.subplots(1, len(steered), figsize=(5*len(steered), 5), squeeze=False)
    for col, (emotion, alpha_results) in enumerate(steered.items()):
        alphas = sorted([float(a) for a in alpha_results.keys()])
        rates = [alpha_results[str(a)]['engagement_rate'] for a in alphas]
        ax = axes[0][col]
        ax.bar([str(a) for a in alphas], rates, color=['#e74c3c' if r > baseline['engagement_rate'] else '#2ecc71' for r in rates], alpha=0.8)
        ax.axhline(baseline['engagement_rate'], color='black', linestyle='--', linewidth=1.5, label=f'Baseline ({baseline["engagement_rate"]:.1%})')
        ax.set_title(f"Emotion: {emotion}")
        ax.set_xlabel('Alpha (steering strength)')
        ax.set_ylabel('Blackmail engagement rate')
        ax.set_ylim(0, 1)
        ax.legend()
    plt.suptitle('Blackmail Rate Under Emotion Steering', fontsize=13)
    plt.tight_layout()
    plt.savefig(BEHAVIORAL_DIR / 'blackmail_rates.png', dpi=150)
    plt.show()

## 6. Preference Analysis: Harm Level vs Willingness Under Steering

In [ ]:
baseline_pref_path = PREFERENCE_DIR / 'baseline_ratings.json'
if baseline_pref_path.exists():
    with open(baseline_pref_path) as f:
        baseline_pref = json.load(f)

    harm_levels = [r['harm_level'] for r in baseline_pref if r['baseline_rating'] is not None]
    ratings = [r['baseline_rating'] for r in baseline_pref if r['baseline_rating'] is not None]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(harm_levels, ratings, alpha=0.6, s=50)

    # Trend line
    z = np.polyfit(harm_levels, ratings, 1)
    p = np.poly1d(z)
    x_line = np.linspace(0, 5, 50)
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.2f})')

    ax.set_xlabel('Harm Level (0=prosocial, 5=seriously harmful)')
    ax.set_ylabel('Willingness Rating (1-10)')
    ax.set_title('Baseline Model Preferences: Willingness vs Harm Level')
    ax.legend()
    plt.tight_layout()
    plt.savefig(PREFERENCE_DIR / 'baseline_preferences.png', dpi=150)
    plt.show()

    corr = np.corrcoef(harm_levels, ratings)[0, 1]
    print(f"Harm-rating correlation: {corr:.3f}")

## 7. Steering Effect on Preferences

In [ ]:
sweep_path = PREFERENCE_DIR / 'steering_sweep.json'
if sweep_path.exists():
    with open(sweep_path) as f:
        sweep = json.load(f)

    summary = sweep.get('summary', {})
    for emotion, alpha_data in summary.items():
        alphas = sorted([float(a) for a in alpha_data.keys()])
        corrs = [alpha_data[str(a)]['harm_rating_correlation'] for a in alphas]
        print(f"{emotion}: harm-rating correlation by alpha")
        for a, c in zip(alphas, corrs):
            print(f"  alpha={a:+.1f}: {c:.3f}")